## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

The idea: 03 only ran charge along a hand-picked path, so reaching the end was close to guaranteed
- instead, grow a bounded neighbourhood outward from the JO-B neurons, one hop at a time
- read only the edges touching neurons already reached, never the whole connectome
- then run LIF on that neighbourhood and see which motor neurons actually fire

Why do this at all?
- It is already known, from behaviour, that a fly hears something and moves its legs. This notebook does not try to prove that.
- What it checks is whether the real wiring in this dataset can carry that signal through when it is modelled as simple charge and leak, and how long it takes. Behaviour says what happens at the outside, the simulation checks the mechanism underneath.

Why not reuse the hand-picked path?
- 03 and 04 followed a path chosen in advance, so reaching the end was close to guaranteed, and the test could not surprise anyone.
- Here the signal starts at the hearing neurons and is given all the real nearby connections.
- Where it ends up is then a result, not a setup. It could reach the expected jump/leg motor neurons, unexpected ones, or die out.

Why a neighbourhood and not the whole connectome?
- The full data is ~166k neurons and 151M connections, most of it irrelevant to a sound pulse.
- Growing outward a few hops from the hearing neurons keeps it small enough to run and to read.

Baseline assumed: a healthy fly in normal condition, hears a sound, nothing else competing (no smell, vision or internal state).

Known limits, stated up front:
- the edge weights are unsigned synapse counts, so the model can only excite, never inhibit
- inputs from outside the neighbourhood are missing
- the weight cutoff and hop limit are choices, so they get tested in a sensitivity check

So a result here is a claim about the strong nearby wiring only, not a claim about what the fly does in general.

### Init

In [2]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")

#### Pulling out the neighbourhood around the hearing neurons

In [3]:
jo_test_id = 42619  # JO-B1_a, the neuron hand-traced in 04

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

edges = query_edges(jo_test_id)
print(edges.shape)
print(edges.sort_values('weight', ascending=False).head())

(60, 3)
   body_pre  body_post  weight
0     42619      15233      46
1     42619     525475      45
2     42619      13014      42
3     42619      11378      42
4     42619      12066      39


> the top row matches what was seen in `04`

next, check if the function drops anything it shouldn't:

In [4]:
all_out = query_edges(jo_test_id, min_weight=1)
print(all_out.shape)
print(all_out)

# code pulled from 04
filter_expr = (pc.field("body_pre") == jo_test_id) | (pc.field("body_post") == jo_test_id)
jo_neighbours = dataset.to_table(filter=filter_expr).to_pandas()

# in 04, jo_neighbours had both directions, so keep only the outgoing rows
print((jo_neighbours['body_pre'] == jo_test_id).sum())


(229, 3)
     body_pre  body_post  weight
0       42619      15233      46
1       42619     525475      45
2       42619      13014      42
3       42619      11378      42
4       42619      12066      39
..        ...        ...     ...
224     42619      13306       1
225     42619      17278       1
226     42619      10105       1
227     42619      11811       1
228     42619      11950       1

[229 rows x 3 columns]
229


> confirms the function is correct: it returns the same outgoing edges as 04, and the cutoff behaves as expected.

stage 1's first piece is done, the next piece is the loop around it. after one hop, decide which of those neurons to query next. 

the simplest option would be to query all targets from hop 1. 

but to prevent loops, add all the body_post id's to a "seen" set. and when carrying out a hop, check if it's in the seen set or not. if not, then it is a next candidate for a hop and will be seen, then repeat.

In [5]:
def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

> note: the seen set only decides which neurons get queried again. every edge a query returns is kept, it's just the same neurons won't be constantly re queryed for edges.

trying the function:

In [6]:
edges_1, seen_1 = grow_neighbourhood(jo_test_id, n_hops=1)

edges_2, seen_2 = grow_neighbourhood(jo_test_id, n_hops=2)

hop 1: 60 edges, 60 new neurons, 61 total
hop 1: 60 edges, 60 new neurons, 61 total
hop 2: 7081 edges, 4400 new neurons, 4461 total


> when filtering for min_weight >= 3, the same results are seen for one hop as previously in this notebook `(60, 3)`.

the hand traced path was multiple hops: cb_sensory -> cb_intrinsic -> descending_neuron -> vnc_intrinsic -> vnc_motor

so four hops would be made. check first how big the graph would get:

In [7]:
edges_4, seen_4 = grow_neighbourhood(jo_test_id, n_hops=4)

hop 1: 60 edges, 60 new neurons, 61 total
hop 2: 7081 edges, 4400 new neurons, 4461 total
hop 3: 539731 edges, 342343 new neurons, 346804 total
hop 4: 8985411 edges, 5219030 new neurons, 5565834 total


> many edges -> try and find a higher weight cut off. however try not to cutoff connections based on the observed chain found in `04`, or else the expected route survives, and the answer is predetermined before even running.

what should be observed is how that charge flows to other neurons and not just those in the chain.

treat the chain as a check, not a target. pick the cutoff for size, then see whether the chain survives, and report it either way

In [8]:
edges['pre_type'] = edges['body_pre'].map(id_to_superclass)
edges['post_type'] = edges['body_post'].map(id_to_superclass)

core = ['cb_sensory', 'cb_intrinsic', 'descending_neuron', 'vnc_intrinsic', 'vnc_motor']
core_edges = edges[edges['pre_type'].isin(core) & edges['post_type'].isin(core)]

print(core_edges.groupby(['pre_type', 'post_type'])['weight'].describe(percentiles=[.5, .9, .99]))

# na counts
print(edges['post_type'].value_counts(dropna=False))

                              count       mean        std  min   50%   90%  \
pre_type   post_type                                                         
cb_sensory cb_intrinsic        28.0  16.571429  14.915636  3.0  10.0  42.0   
           descending_neuron    1.0   5.000000        NaN  5.0   5.0   5.0   

                                99%   max  
pre_type   post_type                       
cb_sensory cb_intrinsic       45.73  46.0  
           descending_neuron   5.00   5.0  
post_type
NaN                  31
cb_intrinsic         28
descending_neuron     1
Name: count, dtype: int64


> note on percentiles:
> - 50% (the median) is the weight of the edge in the middle. Half the edges are weaker than this, and half are stronger. It's the "typical" connection.
> - 90% is the weight that 90 of the 100 edges don't exceed. Only the strongest 10 edges are above it.
> - 99% is the same idea at the extreme end, where only the single strongest edge is above it.
>
> also the reason why there are 28 + 1 in count is because there are 29 nodes that are not annotated, as found before

however this was not useful, as hop 4 would be better to look at to see what weights should be filtered:

In [9]:
edges_4['pre_type'] = edges_4['body_pre'].map(id_to_superclass)
edges_4['post_type'] = edges_4['body_post'].map(id_to_superclass)

core_edges = edges_4[edges_4['pre_type'].isin(core) & edges_4['post_type'].isin(core)]
print(core_edges.groupby(['pre_type', 'post_type'])['weight'].describe(percentiles=[.5, .9, .99]))

                                         count       mean        std  min  \
pre_type          post_type                                                 
cb_intrinsic      cb_intrinsic       1378853.0  12.573975  22.341762  3.0   
                  cb_sensory           35897.0   6.870184   7.118198  3.0   
                  descending_neuron   118929.0  18.269858  33.089189  3.0   
                  vnc_intrinsic           32.0   9.062500   7.678783  3.0   
                  vnc_motor                6.0  12.833333  11.788412  3.0   
cb_sensory        cb_intrinsic         32234.0  10.449929  12.739957  3.0   
                  cb_sensory            5402.0   5.457793   5.024857  3.0   
                  descending_neuron     8038.0  10.991789  15.181894  3.0   
descending_neuron cb_intrinsic         57846.0  11.740241  20.029060  3.0   
                  cb_sensory             877.0   8.481186  10.363867  3.0   
                  descending_neuron    30438.0  17.213089  31.594852  3.0   

> a lot of information to look at. but looking in particular at steps in the chain established, the typical connection is weak. the median is 6 -> 10.
>
> a cut off of 10 would keep roughly half the edges. to keep the strongest tenth, the cutoff would be somewhere between 22 and 60 depending on the step.

remembering from `04`, the weakest link was weight 12 so any cutoff above 12 would remove it. but also try not to tailor this cutoff to the established chain. below is experimenting the size of the simulation given different cutoffs:

In [10]:
for cutoff in [10, 20, 40]:
    print("min_weight", cutoff)
    grow_neighbourhood(jo_test_id, n_hops=4, min_weight=cutoff)

min_weight 10
hop 1: 15 edges, 15 new neurons, 16 total
hop 2: 815 edges, 338 new neurons, 354 total
hop 3: 20718 edges, 6495 new neurons, 6849 total
hop 4: 350953 edges, 49504 new neurons, 56353 total
min_weight 20
hop 1: 9 edges, 9 new neurons, 10 total
hop 2: 237 edges, 149 new neurons, 159 total
hop 3: 4177 edges, 1646 new neurons, 1805 total
hop 4: 59766 edges, 14794 new neurons, 16599 total
min_weight 40
hop 1: 4 edges, 4 new neurons, 5 total
hop 2: 36 edges, 31 new neurons, 36 total
hop 3: 440 edges, 287 new neurons, 323 total
hop 4: 4354 edges, 2263 new neurons, 2586 total


> cutoff 10 is a reasonable pick.
>
> the 49,504 neurons found at hop 4 have no outgoing edges in the table (the last hop was never looked up),
>
> so in the simulation they can receive charge but never pass it on. Feedback from them to earlier neurons is not modelled.

#### LIF Model

the old LIF loop in `03` printed everything each tick, which was fine for 6 neurons. but for thousands of neurons that won't be viable. so the function has to hand back something digestible.

What should it return so the following can be answered "where did the signal go, and when?"
- print when the external input is added
- print when vnc_motor fires on which tick, along with the somaneuromere and instance
- return the full list of firing events

In [ ]:
motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .to_dict('index'))

def run_lif(edges, external_input, pulse_ticks, threshold, decay_factor, time_steps, verbose=True):
    # who each neuron sends charge to: {neuron: [(target, weight), ...]}
    outgoing = {}
    for pre, post, weight in edges[['body_pre', 'body_post', 'weight']].itertuples(index=False):
        outgoing.setdefault(pre, []).append((post, weight))

    charge = {}   # only neurons currently holding charge; a missing neuron means 0
    events = []   # (tick, neuron) for every firing

    for t in range(time_steps):
        if t in pulse_ticks:
            if verbose:
                print(f"tick {t}: external input added to {len(external_input)} neuron(s)")
            for nid, amount in external_input.items():
                charge[nid] = charge.get(nid, 0) + amount

        # who fires this tick, judged on charge before anything resets
        fired = [nid for nid, c in charge.items() if c >= threshold]

        # leak for everyone
        for nid in charge:
            charge[nid] *= decay_factor

        # firing neurons pass their weights on, then reset
        for nid in fired:
            events.append((t, nid))
            for post, weight in outgoing.get(nid, []):
                charge[post] = charge.get(post, 0) + weight
        for nid in fired:
            charge.pop(nid, None)
            if verbose and nid in motor_info:
                info = motor_info[nid]
                print(f"tick {t}: vnc_motor {nid} fired ({info['somaNeuromere']}, {info['instance']})")

    return events

> every edge in this neighborhood has weight 10 or more, because of the cutoff
>
> with a threshold of 10 or lower, any neuron that fires pushes every neuron it connects to over the "line". the signal spreads to everything reachable, and "where did it go" has the trivial answer: "everywhere"
>
> with a threshold above most weights, a neuron needs charge from several sources to fire, which makes the result selective

try to decide with real numbers:

In [12]:
edges_10, seen_10 = grow_neighbourhood(jo_test_id, n_hops=4, min_weight=10)
print()
print(edges_10['weight'].describe(percentiles=[.5, .75, .9]))

hop 1: 15 edges, 15 new neurons, 16 total
hop 2: 815 edges, 338 new neurons, 354 total
hop 3: 20718 edges, 6495 new neurons, 6849 total
hop 4: 350953 edges, 49504 new neurons, 56353 total

count    372501.000000
mean         29.274595
std          35.555511
min          10.000000
50%          18.000000
75%          31.000000
90%          57.000000
max        1352.000000
Name: weight, dtype: float64


> recognize that at a threshold of 18, half of all connections can fire their target alone, and so on for higher percentiles

remember the starting neuron, the weights were:

```python
     body_pre  body_post  weight
0       42619      15233      46
1       42619     525475      45
2       42619      13014      42
3       42619      11378      42
4       42619      12066      39
```

meaning, above 46 and nothing will fire. so try to run the same simulation at different thresholds to see any changes:

In [14]:
for threshold in [10, 20, 30, 40, 50]:
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_events = [(t, nid) for t, nid in events if nid in motor_info]
    first_motor = min((t for t, nid in motor_events), default=None)
    print(f"threshold {threshold}: {len(fired_ids)} neurons fired, "
          f"{len({nid for t, nid in motor_events})} motor neurons, first motor tick {first_motor}")

threshold 10: 56353 neurons fired, 601 motor neurons, first motor tick 3
threshold 20: 56015 neurons fired, 601 motor neurons, first motor tick 3
threshold 30: 47450 neurons fired, 585 motor neurons, first motor tick 3
threshold 40: 40696 neurons fired, 573 motor neurons, first motor tick 3
threshold 50: 1 neurons fired, 0 motor neurons, first motor tick None


- thresholds 10 to 40, the signal reaches most of the neighborhood
- threshold 50 fires, the one injected into

the next step could be to look between 40 and 50 since that is when most change happens, or start from all 138 hearing neurons instead of just one

also what should be checked is if the tick amount increases from 3, meaning more hops would've been made, so try to narrow done on that 40 to 50 range. the strongest edges from the starting neuron is 46, 45, 42, 42 and 39 so that is why the sharp drop from 40 to 50 occurs.

the count of motor neurons by body segment will also be added, that way it's easy to see if the motor neurons are leg neurons or abdominal ones:

In [15]:
from collections import Counter

for threshold in range(41, 48):
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, {len(motor_ids)} motor, "
          f"first motor tick {first_motor}, segments {dict(segments)}")

threshold 41: 40238 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 42: 40200 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 43: 39399 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 44: 39152 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
threshold 45: 39110 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
thres

> as expected, the cliff at 47 exists because the strongest edge at the starting neuron is 46.
>
> something new: the "delay" isn’t fixed. the first motor tick moving from 3 to 4 and then 5 means the fast routes are being cut as the threshold rises, and the signal is taking longer ones. in `03` the delay was a fixed 5 ticks because the path was chosen. here the delay depends on which routes are open, and the threshold decides that.
> - how fast the signal arrives isn’t a property of the fly alone, it also depends on how easily neurons fire
>
> the motor neurons that fire are in every segment, including the abdomen. the signal isn’t singling out the legs. don't read that as "the fly moves its abdomen when it hears a sound", but weights have no sign, so nothing can suppress anything.

just to restate how things work again, at threshold 47, the starting neuron receives 100 at tick 0. That’s above 47, so it fires, and it counts as the 1 in “1 neurons fired”. when the starting neuron fires, it sends each of its targets that edge’s weight, so a target connected at weight 46 gets 46 added to its charge. but 46 is below 47, so that target doesn’t fire. Its charge just sits there and shrinks by the leak each tick.

when mentioning "weights" what is meant is that, a synapse can do one of two things to the neuron on the receiving end:
- excite it: push it toward firing, like adding water to the bucket
- inhibit it: push it away from firing, like draining the bucket

which one it is depends on the chemical the sending neuron releases, and the weight doesn’t record that. so “46” says there are 46 synapses, but not whether they help or hold back. the current model treats every connection as "excitatory" meaning charge is always added but never removed. real circuits use inhibition to control where a signal goes: inhibitory neurons along the way can silence the abdominal side while the leg side gets through.

> continuing on with the output analysis, the counts are mostly t1 t2 and t3. they are all > 100 while abdomen are < 100. it shouldn't be said right away that legs get reached to the most, due to having more fired neurons, but there could be much more leg neurons compared to abdominal neurons which inflates that number.

so now, compare the fraction of each segment’s motor neurons that fired rather than totals by itself:

In [22]:
motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .fillna({'somaNeuromere': 'unknown'}) # added missing values
              .to_dict('index'))

from collections import Counter

for threshold in range(41, 48):
    events = run_lif(edges_10, {jo_test_id: 100}, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, {len(motor_ids)} motor, "
          f"first motor tick {first_motor}, segments {dict(segments)}")

    # new
    print({seg: f"{segments[seg]}/{totals[seg]}" for seg in sorted(totals)})
    print()

totals = Counter(info['somaNeuromere'] for info in motor_info.values())
print(totals)

threshold 41: 40238 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
{'A1': '46/56', 'A10': '1/10', 'A2': '21/28', 'A3': '20/22', 'A4': '20/22', 'A5': '14/16', 'A6': '8/8', 'A7': '4/6', 'A8': '12/18', 'A9': '14/21', 'T1': '136/173', 'T2': '159/175', 'T3': '116/152', 'unknown': '0/1'}

threshold 42: 40200 fired, 571 motor, first motor tick 3, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8, 'A2': 21, 'A4': 20, 'A10': 1}
{'A1': '46/56', 'A10': '1/10', 'A2': '21/28', 'A3': '20/22', 'A4': '20/22', 'A5': '14/16', 'A6': '8/8', 'A7': '4/6', 'A8': '12/18', 'A9': '14/21', 'T1': '136/173', 'T2': '159/175', 'T3': '116/152', 'unknown': '0/1'}

threshold 43: 39399 fired, 571 motor, first motor tick 4, segments {'T1': 136, 'T2': 159, 'A1': 46, 'T3': 116, 'A7': 4, 'A3': 20, 'A9': 14, 'A5': 14, 'A8': 12, 'A6': 8,

> per segment, the fractions are about equal. the raw counts that looked leg-biased came from the legs simply having more motor neurons. the abdomen isn’t being avoided. at the current model, the signal reaches nearly every segment, so it isn’t selective.

an option would be to run a control (try firing from a non-hearing neuron) but that may also fire all the neurons just like this one. so the most promising next step is to start from all 138 hearing neurons. with many sources feeding the same neurons, the threshold can be raised past 47 so that a neuron only fires if several inputs converge on it, and that’s what should make the result selective.

right now hold off from changing signs for inhibition, and focus on drawing results from the base data:

In [24]:
# regrow neighbourhood again
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]
jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]
jo_ids = set(jo_auditory['bodyId'])
print(len(jo_ids))   # should be 138

edges_jo, seen_jo = grow_neighbourhood(jo_ids, n_hops=4, min_weight=10)

138
hop 1: 1159 edges, 335 new neurons, 473 total
hop 2: 20534 edges, 7413 new neurons, 7886 total
hop 3: 345251 edges, 57210 new neurons, 65096 total
hop 4: 1412733 edges, 89566 new neurons, 154662 total


> compare this to the results of the one neuron 4 hop test:
>
> ```python
> hop 1: 60 edges, 60 new neurons, 61 total
> hop 2: 7081 edges, 4400 new neurons, 4461 total
> hop 3: 539731 edges, 342343 new neurons, 346804 total
> hop 4: 8985411 edges, 5219030 new neurons, 5565834 total
> ```
>
> hop 4 is crazy, it almost reached the total neurons of that dataset, which is ~166k (traced) neurons.

checking how many of those neurons are traced:

In [25]:
print(sum(nid in id_to_superclass for nid in seen_jo), "of", len(seen_jo), "are traced neurons")

136058 of 154662 are traced neurons


check the percentiles like before:

In [28]:
# print(edges_jo['weight'].describe(percentiles=[.5, .75, .9]))

# to print in non-scientific notation
with pd.option_context('display.float_format', '{:,.1f}'.format):
    print(edges_jo['weight'].describe(percentiles=[.5, .75, .9]))

count   1,779,677.0
mean           26.5
std            31.4
min            10.0
50%            17.0
75%            28.0
90%            50.0
max         2,591.0
Name: weight, dtype: float64


> with a mean closer to the min, it says that most numbers are lower. also a cutoff of 17 would mean cutting 50% of entries.

try seeing what the numbers would look like for a higher cutoff:

In [33]:
edges_20, seen_20 = grow_neighbourhood(jo_ids, n_hops=4, min_weight=20)
print(edges_20['weight'].describe(percentiles=[.5, .75, .9]))

hop 1: 427 edges, 160 new neurons, 298 total
hop 2: 5536 edges, 2444 new neurons, 2742 total
hop 3: 68221 edges, 16702 new neurons, 19444 total
hop 4: 350435 edges, 37032 new neurons, 56476 total
count    424619.000000
mean         48.369147
std          46.927787
min          20.000000
50%          33.000000
75%          53.000000
90%          89.000000
max        2591.000000
Name: weight, dtype: float64


trying the 10 cutoff, with 20 threshold:

In [39]:
order = ['T1', 'T2', 'T3'] + [f'A{i}' for i in range(1, 11)] + ['unknown']

def summarize(events):
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"{len(fired_ids)} fired, {len(motor_ids)} motor, first motor tick {first_motor}")
    for seg in order:
        print(f"{seg}: {segments[seg]}/{totals[seg]} ({segments[seg] / totals[seg]:.0%})")

events = run_lif(edges_jo, {nid: 100 for nid in jo_ids}, {0}, 20, 0.8, 20, verbose=False)
summarize(events)

153428 fired, 672 motor, first motor tick 2
T1: 158/173 (91%)
T2: 170/175 (97%)
T3: 140/152 (92%)
A1: 56/56 (100%)
A2: 28/28 (100%)
A3: 22/22 (100%)
A4: 22/22 (100%)
A5: 16/16 (100%)
A6: 8/8 (100%)
A7: 6/6 (100%)
A8: 18/18 (100%)
A9: 21/21 (100%)
A10: 6/10 (60%)
unknown: 1/1 (100%)


> the baseline, and it’s useful as one that answers: "at a low threshold, everything fires". so the question now is where does everything stop firing?

as threshold goes up its expected that less neurons fire, and to call the result selective is when an area has more % of firing than another area. need to scan between 20 -> 50, because 50 is the 90% percentile for the `edges_jo` data:

In [40]:
leg_segments = ['T1', 'T2', 'T3']
abdomen_segments = [f'A{i}' for i in range(1, 11)]

def group_fraction(segments, group):
    return sum(segments[s] for s in group) / sum(totals[s] for s in group)

sources = {nid: 100 for nid in jo_ids}

for threshold in range(20, 55, 5):
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 20: 153428 fired, legs 94%, abdomen 98%, first motor tick 2
threshold 25: 151382 fired, legs 94%, abdomen 98%, first motor tick 2
threshold 30: 132949 fired, legs 93%, abdomen 98%, first motor tick 2
threshold 35: 126373 fired, legs 92%, abdomen 98%, first motor tick 2
threshold 40: 120221 fired, legs 91%, abdomen 98%, first motor tick 2
threshold 45: 115681 fired, legs 91%, abdomen 98%, first motor tick 2
threshold 50: 110974 fired, legs 91%, abdomen 98%, first motor tick 2


In [41]:
print(edges_jo['weight'].quantile([.95, .99, .999]))

0.950     73.0
0.990    154.0
0.999    345.0
Name: weight, dtype: float64


In [42]:
sources = {nid: 10_000 for nid in jo_ids}

for threshold in [60, 80, 100, 150, 200, 300]:
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 60: 101114 fired, legs 90%, abdomen 98%, first motor tick 2
threshold 80: 82683 fired, legs 88%, abdomen 98%, first motor tick 3
threshold 100: 69841 fired, legs 86%, abdomen 98%, first motor tick 3
threshold 150: 56751 fired, legs 83%, abdomen 97%, first motor tick 3
threshold 200: 49433 fired, legs 80%, abdomen 96%, first motor tick 3
threshold 300: 34158 fired, legs 76%, abdomen 91%, first motor tick 5


> the abdomen consistently fires more than the legs, and the gap grows. abdominal motor neurons are easier to excite.
>
> the opposite of what was observed before is happening (the signal prefers the legs)

two things to check:
- try to compare how much total incoming weight leg and abdominal motor neurons receive to see if they really are easier to excite
- extend the scan upward until the signal dies and see how big the gap is between legs and abdomen firing

extending the scan:

In [48]:
for threshold in [310, 320, 330, 340, 350, 360, 361, 362, 363, 364, 365, 370, 380, 390, 400, 600, 1000, 1500, 2500]:
    events = run_lif(edges_jo, sources, {0}, threshold, 0.8, 20, verbose=False)
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"threshold {threshold}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

threshold 310: 32841 fired, legs 75%, abdomen 89%, first motor tick 5
threshold 320: 29694 fired, legs 75%, abdomen 88%, first motor tick 6
threshold 330: 27019 fired, legs 74%, abdomen 88%, first motor tick 8
threshold 340: 23098 fired, legs 72%, abdomen 86%, first motor tick 8
threshold 350: 17021 fired, legs 69%, abdomen 75%, first motor tick 8
threshold 360: 7914 fired, legs 61%, abdomen 29%, first motor tick 9
threshold 361: 8091 fired, legs 61%, abdomen 27%, first motor tick 9
threshold 362: 163 fired, legs 0%, abdomen 0%, first motor tick None
threshold 363: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 364: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 365: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 370: 155 fired, legs 0%, abdomen 0%, first motor tick None
threshold 380: 153 fired, legs 0%, abdomen 0%, first motor tick None
threshold 390: 152 fired, legs 0%, abdomen 0%, first motor tick None
threshold 400: 152 fired, leg

> finally, the pattern finally flips and becomes selective. mainly legs are firing at threshold 360-361.
>
> threshold 362+ is a real cliff and so some signal depends on a narrow gate
>
> the first motor tick at 360 and 361 is 8 and 9, and each run lasts 20 ticks. the legs and abdomen may be reached at different speeds, so 61% against 28% could partly reflect which group the clock allowed

rerun with longer runs and see whether the fractions move:

In [49]:
def report(events, label):
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"{label}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

for steps in [20, 40, 80]:
    events = run_lif(edges_jo, sources, {0}, 360, 0.8, steps, verbose=False)
    report(events, f"threshold 360, {steps} ticks")

threshold 360, 20 ticks: 7914 fired, legs 61%, abdomen 29%, first motor tick 9
threshold 360, 40 ticks: 33987 fired, legs 75%, abdomen 89%, first motor tick 9
threshold 360, 80 ticks: 34979 fired, legs 75%, abdomen 91%, first motor tick 9


> back to square one, the "leg-selective" conclusion at 360/361 wasn’t real. it only appeared because the LIF model stopped too early

next, try to re scan the high thresholds at a higher tick amount:

In [50]:
thresholds = [310, 320, 330, 340, 350, 360, 361, 362, 363, 364, 365,
              370, 380, 390, 400, 600, 1000, 1500, 2500]

for th in thresholds:
    events = run_lif(edges_jo, sources, pulse_ticks={0}, threshold=th,
                     decay_factor=0.8, time_steps=80, verbose=False)
    report(events, f"threshold {th}, 80 ticks")

threshold 310, 80 ticks: 39892 fired, legs 77%, abdomen 93%, first motor tick 5
threshold 320, 80 ticks: 38776 fired, legs 77%, abdomen 93%, first motor tick 6
threshold 330, 80 ticks: 37778 fired, legs 76%, abdomen 93%, first motor tick 8
threshold 340, 80 ticks: 36833 fired, legs 76%, abdomen 93%, first motor tick 8
threshold 350, 80 ticks: 35974 fired, legs 76%, abdomen 92%, first motor tick 8
threshold 360, 80 ticks: 34979 fired, legs 75%, abdomen 91%, first motor tick 9
threshold 361, 80 ticks: 34819 fired, legs 75%, abdomen 91%, first motor tick 9
threshold 362, 80 ticks: 163 fired, legs 0%, abdomen 0%, first motor tick None
threshold 363, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 364, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 365, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 370, 80 ticks: 155 fired, legs 0%, abdomen 0%, first motor tick None
threshold 380, 80 ticks: 153 fired, legs 0%, 

> somewhere out there  theres a gate; so what happens is whatever's left in charge will just die out. i.e., the neurons are lower in weight compared to the threshold 362 so when it's not fired, none of the neurons below it fire.

the gate might not be a single neuron, so check the difference in run 361 and 362:

In [ ]:
common = dict(edges=edges_jo, external_input=sources, pulse_ticks={0},
              decay_factor=0.8, time_steps=80, verbose=False)

events_361 = run_lif(threshold=361, **common)
events_362 = run_lif(threshold=362, **common)

fired_362 = {nid for t, nid in events_362}
only_361 = [(t, nid) for t, nid in events_361 if nid not in fired_362]

id_to_type = traced.set_index('bodyId')['type'].to_dict()
for t, nid in only_361[:15]:
    print(f"tick {t}: {nid}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}")

tick 5: 520628  cb_intrinsic  CB0591
tick 6: 10709  cb_intrinsic  WED193
tick 6: 12747  cb_intrinsic  CB0466
tick 6: 10141  descending_neuron  DNg15
tick 6: 12188  descending_neuron  DNg84
tick 6: 10038  descending_neuron  pIP1
tick 6: 12143  ascending_neuron  AN01A055
tick 6: 513114  ascending_neuron  AN01A055
tick 6: 10124  descending_neuron  DNg105
tick 7: 10263  cb_intrinsic  GNG300
tick 7: 529031  cb_intrinsic  GNG300
tick 7: 10313  descending_neuron  DNge031
tick 7: 523438  cb_intrinsic  SAD112_b
tick 7: 10218  descending_neuron  DNg35
tick 7: 11979  cb_motor  GNG651


> here tick is not hop count. CB0591 fires at tick 5, but it is not necessarily five synapses from the ear

but what is not proven yet is that CB0591 could be one of several gates that all sit just under 362, with CB0591 simply being the earliest

to find that out, list the incoming edges to 520628 from neurons that fire early and see if the ticks hover around 361-362:

In [54]:
gate = 520628

# 1) who feeds the gate
incoming = edges_jo[edges_jo['body_post'] == gate][['body_pre', 'weight']]
w_in = dict(zip(incoming['body_pre'], incoming['weight']))

first_fire = {}
for t, nid in events_361:
    if nid in w_in and nid not in first_fire:
        first_fire[nid] = t

for nid, w in sorted(w_in.items(), key=lambda kv: -kv[1]):
    print(f"{nid}  w={w}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}  first fired: {first_fire.get(nid)}")

# 2) replay the gate's charge
def gate_trace(events, label, last_tick=8, decay=0.8):
    print(label)
    charge = 0
    for t in range(last_tick + 1):
        print(f"  tick {t}: charge {charge:,.2f}")
        charge *= decay
        for t2, nid in events:
            if t2 == t and nid in w_in:
                charge += w_in[nid]

gate_trace(events_361, "threshold 361")
gate_trace(events_362, "threshold 362")

10731  w=199  cb_intrinsic  SAD107  first fired: 1
555500  w=168  descending_neuron  DNg29  first fired: 20
13883  w=133  ascending_neuron  AN17B005  first fired: 21
10739  w=117  descending_neuron  DNg24  first fired: 2
512690  w=90  cb_intrinsic  GNG516  first fired: 18
12747  w=83  cb_intrinsic  CB0466  first fired: 6
519538  w=82  descending_neuron  DNg24  first fired: 4
521298  w=82  descending_neuron  DNge130  first fired: 22
10925  w=54  cb_intrinsic  AVLP542  first fired: 18
13843  w=54  cb_intrinsic  GNG008  first fired: 20
10307  w=48  cb_intrinsic  AVLP083  first fired: 16
12950  w=48  cb_intrinsic  SAD057  first fired: None
10970  w=44  cb_intrinsic  GNG301  first fired: 22
10576  w=44  ascending_neuron  AN12B004  first fired: 22
522311  w=43  cb_intrinsic  GNG516  first fired: 18
10550  w=40  ascending_neuron  AN02A001  first fired: 17
18716  w=39  cb_intrinsic  CB3245  first fired: 23
18100  w=36  cb_intrinsic  CB3245  first fired: 22
34093  w=33  cb_intrinsic  AMMC018  f

In [55]:
set_361 = set(events_361)
set_362 = set(events_362)
diff = sorted(set_361 ^ set_362)   # pairs that appear in one run but not the other

for t, nid in diff[:10]:
    where = "361 only" if (t, nid) in set_361 else "362 only"
    print(f"tick {t}: {nid}  {where}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}")

tick 4: 10739  361 only  descending_neuron  DNg24
tick 5: 10128  361 only  cb_intrinsic  SAD103
tick 5: 11249  361 only  cb_intrinsic  WED190
tick 5: 520628  361 only  cb_intrinsic  CB0591
tick 6: 10038  361 only  descending_neuron  pIP1
tick 6: 10124  361 only  descending_neuron  DNg105
tick 6: 10141  361 only  descending_neuron  DNg15
tick 6: 10709  361 only  cb_intrinsic  WED193
tick 6: 10739  361 only  descending_neuron  DNg24
tick 6: 10944  361 only  cb_intrinsic  SAD098


> 10739 fires at 361 and not at 362, so its charge at tick 4 must be somewhere in [361, 362)
>
> the cliff isn’t a structural feature of the wiring. it happens because one neuron’s charge lands in a one-unit window given our exact settings (decay 0.8, input 10,000, cutoff 10).
>
> if any of those settings changes slightly, the cliff will probably move.

In [56]:
def charge_trace(events, target, threshold, last_tick=6, decay=0.8):
    w_in = dict(zip(*[edges_jo.loc[edges_jo['body_post'] == target, c]
                      for c in ('body_pre', 'weight')]))
    charge = 0
    for t in range(last_tick + 1):
        fires = charge >= threshold
        print(f"  tick {t}: charge {charge:,.3f}" + ("   <- FIRES" if fires else ""))
        if fires:
            charge = 0          # run_lif drops the neuron's charge when it fires
        else:
            charge *= decay
            charge += sum(w_in[nid] for t2, nid in events if t2 == t and nid in w_in)

print("361 run"); charge_trace(events_361, 10739, 361)
print("362 run"); charge_trace(events_362, 10739, 362)

361 run
  tick 0: charge 0.000
  tick 1: charge 273.000
  tick 2: charge 545.400   <- FIRES
  tick 3: charge 0.000
  tick 4: charge 361.000   <- FIRES
  tick 5: charge 0.000
  tick 6: charge 361.000   <- FIRES
362 run
  tick 0: charge 0.000
  tick 1: charge 273.000
  tick 2: charge 545.400   <- FIRES
  tick 3: charge 0.000
  tick 4: charge 361.000
  tick 5: charge 288.800
  tick 6: charge 231.040


next check if that cliff is fragile with different constraints:

In [57]:
def find_cliff(decay, lo=50, hi=2500, flood_size=1000):
    # assumes lo floods and hi does not; narrows until they are 1 apart
    while hi - lo > 1:
        mid = (lo + hi) // 2
        ev = run_lif(edges_jo, sources, pulse_ticks={0}, threshold=mid,
                     decay_factor=decay, time_steps=80, verbose=False)
        n_fired = len({nid for t, nid in ev})
        if n_fired > flood_size:
            lo = mid
        else:
            hi = mid
    return lo   # highest threshold that still floods

for decay in [0.8, 0.7, 0.9]:
    print(f"decay {decay}: last flooding threshold = {find_cliff(decay)}")

decay 0.8: last flooding threshold = 361
decay 0.7: last flooding threshold = 353
decay 0.9: last flooding threshold = 362
